# 第16講：データ分析チートシート

前期と同じ高校の擬似データ（生徒677人）で、第01講から第15講のコードを最初から最後まで動かす。
新しい文法は足さない。点数と体格が、性別・学年・クラスのどれと結びついているかを確認する。


## 0. 準備

最初に1回だけ実行する。グラフの軸に日本語を出すため、Noto Sans JP を登録する。


In [ ]:
import piplite
import pyodide
await piplite.install("seaborn")

import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import font_manager as fm
%matplotlib inline

from pyodide.http import pyfetch
resp = await pyfetch(
    "https://raw.githubusercontent.com/google/fonts/main/ofl/notosansjp/NotoSansJP%5Bwght%5D.ttf"
)
with open("NotoSansJP.ttf", "wb") as f:
    f.write(await resp.bytes())
fm.fontManager.addfont("NotoSansJP.ttf")

sns.set_theme(style="darkgrid", font="Noto Sans JP")


## 1. 読み込む

`mhigh.csv` は1人1行の横断面データ。欠測と氏名の重複を見てから、列名を揃える。
第03講では合計列を `sum` と呼んだ。`sum` はメソッド名とかぶるので、今日は `total` にする。


In [ ]:
url = "https://raw.githubusercontent.com/ChungWookyung/kg-jupyterlite-data-analysis/main/content/DATA/mhigh.csv"
data = pd.read_csv(pyodide.http.open_url(url))
data.shape


In [ ]:
data.head()


In [ ]:
data.isna().sum().sum()


In [ ]:
data["氏名"].duplicated().sum()


In [ ]:
data.columns = ["name", "katakana", "sex", "grade",
                "japanese", "math", "total", "height", "weight", "class"]
data["sex"] = data["sex"].replace({"男": "Male", "女": "Female"})
data.head()


欠測は0件、氏名の重複も0件。677行・10列。カタカナ列の元の名前は `氏名.カタカナ.` だった。


## 2. 行を取る

`loc` は列名、`iloc` は位置（0始まり、終わりは含まない）。条件は括弧で包み、かつは `&`、またはは `|`。


In [ ]:
data.loc[data["name"] == "住田佳織", ["grade", "class", "japanese", "math"]]


In [ ]:
data.iloc[:5, 4:7]


In [ ]:
follow = data[(data["grade"] == 3) & (data["math"] >= 80)]
follow.shape


In [ ]:
low = data[(data["math"] < 40) | (data["japanese"] < 40)]
low.shape


3年生かつ数学80点以上は26人。数学か国語の少なくとも一方が40点未満は210人。


## 3. 集計する

平均を取る列は明示する。列を指定しない `groupby().mean()` は、pandas 2 では文字列列が混ざって失敗する。


In [ ]:
data["math"].mean()


In [ ]:
data["math"].std()


In [ ]:
data["sex"].value_counts()


In [ ]:
data["grade"].value_counts().sort_index()


In [ ]:
data["class"].value_counts().sort_index()


In [ ]:
data.groupby("sex")[["japanese", "math", "height"]].mean().round(1)


In [ ]:
data.groupby("class")["math"].mean().round(1)


In [ ]:
data.groupby("grade")["math"].mean().round(1)


In [ ]:
data.pivot_table(index="grade", columns="class", values="math").round(1)


In [ ]:
(data["total"] == data["japanese"] + data["math"]).all()


数学の平均は56.4、標準偏差は15.1。性別の点数差は1点余りで、身長差は約13cm。
クラスの数学平均は A の77.9から H の35.8へ下がる。学年の差は1年55.6、2年55.3、3年58.3と小さい。
成績合計は677人とも国語と数学の和なので、別の科目としては扱わない。


## 4. 列を足す、表を繋ぐ

`def` と `if` は1点の区分に使い、`apply` で列にする。
成績表と身体測定が別ファイルだった想定で `merge` し、学年ごとのファイルを `concat` で積み戻す。


In [ ]:
def band(score):
    if score >= 80:
        return "高"
    elif score >= 60:
        return "中"
    return "低"

data["math_band"] = data["math"].apply(band)
data["math_band"].value_counts()


In [ ]:
data["bmi"] = data["weight"] / (data["height"] / 100) ** 2
data["bmi"].agg(["mean", "std"]).round(2)


In [ ]:
data.groupby("sex")["bmi"].mean().round(2)


In [ ]:
mu = data["math"].mean()
sd = data["math"].std()
data["math_t"] = 50 + 10 * (data["math"] - mu) / sd
data["math_t"].mean().round(1)


In [ ]:
scores = data[["name", "sex", "grade", "class", "japanese", "math", "total"]]
body = data[["name", "height", "weight"]]
merged = pd.merge(scores, body, on="name")
merged.shape


In [ ]:
lower = data[data["grade"] < 3]
third = data[data["grade"] == 3]
pd.concat([lower, third]).shape


数学の3区分は、高50人・中224人・低403人。平均56点のテストで80点以上を「高」にすると、低が過半数になる。
BMI の平均は21.3、標準偏差は0.57。男女差はほとんどない。
氏名の重複が0なので `merge` の行数は677のまま。実在の名簿では同姓同名があるので、キーは学籍番号にする。


## 5. グラフにする

分布は `histplot`、2つの量の関係は `scatterplot`、グループの平均は `barplot`。
このファイルに日付列はないので、折れ線（第05講）は使わない。


### 5.1 数学の分布（性別）

In [ ]:
sns.histplot(data=data, x="math", hue="sex", kde=True)
plt.xlabel("数学")
plt.ylabel("人数")


男女の山はほぼ同じ位置にある。平均の差が1点余りであることと一致する。

### 5.2 身長の分布（性別）

In [ ]:
sns.histplot(data=data, x="height", hue="sex", kde=True)
plt.xlabel("身長 (cm)")
plt.ylabel("人数")


女子は160cm付近、男子は170cm付近に山がある。

### 5.3 体重と身長

In [ ]:
sns.scatterplot(data=data, x="weight", y="height", hue="sex")
plt.xlabel("体重 (kg)")
plt.ylabel("身長 (cm)")


右上がりで、男女はほぼ別の帯に分かれる。

### 5.4 国語と数学（クラス）

In [ ]:
sns.scatterplot(data=data, x="japanese", y="math", hue="class",
                hue_order=list("ABCDEFGH"))
plt.xlabel("国語")
plt.ylabel("数学")


A は右上、H は左下に寄る。国語が高い生徒は数学も高い。

### 5.5 クラス別の数学の平均

In [ ]:
sns.barplot(data=data, x="class", y="math", order=list("ABCDEFGH"))
plt.xlabel("クラス")
plt.ylabel("数学の平均")


棒は平均、上の線はばらつきの目安。A から H へ階段状に下がる。

### 5.6 国語と数学を同じ棒グラフに

`melt` で科目を縦長にすると、`hue` で色を分けられる。


In [ ]:
long = data.melt(id_vars="class", value_vars=["japanese", "math"],
                 var_name="subject", value_name="score")
long["subject"] = long["subject"].replace({"japanese": "国語", "math": "数学"})
sns.barplot(data=long, x="class", y="score", hue="subject",
            order=list("ABCDEFGH"))


どのクラスでも数学が国語より少し高い。クラス間の落差の方が大きい。

## 6. 相関を数値にする

成績合計は国語と数学の和なので、相関の計算には入れない。


In [ ]:
cols = ["japanese", "math", "height", "weight"]
data[cols].corr().round(2)


国語と数学は 0.52。身長と体重は 0.97。数学と身長は -0.02。
クラスの平均点は A から H へ下がるが、クラス分けが点数を上げたのか、高得点の生徒を A に入れたのかは、この表だけでは分からない。


## 実習①：読み込み・絞り込み・基礎統計

上のデモとは別に、このセルから書き始める。

1. `mhigh.csv` を読み込み、列名を name, katakana, sex, grade, japanese, math, total, height, weight, class に変え、性別を Male / Female に置き換える
2. 欠測の合計件数と、氏名（`name`）の重複件数を数える
3. 3年生かつ数学が80点以上の生徒は何人か
4. 性別ごとの国語・数学・身長の平均を求める


## 実習②：新しい列と表の結合

実習①で作った `data` を使う。

1. BMI の列を作る（体重 ÷（身長をメートルに直した値の2乗））
2. 数学の偏差値の列を作る（50 + 10 ×（数学 − 数学の平均）÷ 数学の標準偏差）
3. 学年を行、クラスを列、値を数学の平均にして `pivot_table` を作る
4. 成績の列と身体測定の列を氏名で `merge` し、行数が 677 のまま戻ることを確認する


## 実習③：グラフと読み取り

実習①で作った `data` を使う。

1. 数学のヒストグラムを性別で色分けし、`kde=True` を付ける
2. 体重を横軸、身長を縦軸にした散布図を、性別で色分けする
3. クラスごとの数学の平均を棒グラフで描く
4. 図から、数学の点数とより強く結びついているのは性別かクラスか、理由とともに1〜2文で書く
